In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/cleaned/supply_chain_cleaned.csv')

print("Shape:", df.shape)

# Normalize helper
def normalize(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([50] * len(series), index=series.index)
    return ((series - min_val) / (max_val - min_val) * 100).round(2)

def risk_label(score):
    if score >= 70:
        return 'HIGH RISK'
    elif score >= 40:
        return 'MEDIUM RISK'
    else:
        return 'LOW RISK'

print("Helper functions ready.")

Shape: (180519, 29)
Helper functions ready.


In [3]:
# ─────────────────────────────────────────────
# RISK TREND — last 3 months per region
# ─────────────────────────────────────────────

df['order_month'] = pd.to_datetime(df['order_month'])
last_3_months = sorted(df['order_month'].unique())[-3:]
recent_df = df[df['order_month'].isin(last_3_months)]

print("Last 3 months used for trend:")
print(last_3_months)
print("Rows in last 3 months:", len(recent_df))

trend_df = recent_df.groupby(['order_region', 'order_month']).agg(
    total_orders=('order_id', 'count'),
    late_orders=('late_delivery_risk', 'sum')
).reset_index()

trend_df['late_rate_pct'] = (trend_df['late_orders'] / trend_df['total_orders'] * 100).round(2)

trend_pivot = trend_df.pivot_table(
    index='order_region', columns='order_month', values='late_rate_pct'
).reset_index()

trend_pivot.columns = [str(c) for c in trend_pivot.columns]
month_cols = [c for c in trend_pivot.columns if c != 'order_region']

trend_pivot['late_rate_oldest'] = trend_pivot[month_cols[0]]
trend_pivot['late_rate_latest'] = trend_pivot[month_cols[-1]]

trend_pivot['late_rate_change_pct'] = (
    (trend_pivot['late_rate_latest'] - trend_pivot['late_rate_oldest'])
    / trend_pivot['late_rate_oldest'].replace(0, np.nan) * 100
).round(2)

trend_pivot['risk_trend'] = trend_pivot['late_rate_change_pct'].apply(
    lambda x: 'Worsening ↑' if x > 10 else ('Improving ↓' if x < -10 else 'Stable →')
)

trend_pivot = trend_pivot[['order_region', 'late_rate_change_pct', 'risk_trend']]

print("\nRisk trend sample:")
print(trend_pivot.head(10))

Last 3 months used for trend:
[Timestamp('2017-11-01 00:00:00'), Timestamp('2017-12-01 00:00:00'), Timestamp('2018-01-01 00:00:00')]
Rows in last 3 months: 6302

Risk trend sample:
      order_region  late_rate_change_pct risk_trend
0     Eastern Asia                  5.65   Stable →
1  Northern Europe                   NaN   Stable →
2          Oceania                  3.08   Stable →
3       South Asia                 -0.70   Stable →
4   Southeast Asia                  2.58   Stable →
5  Southern Europe                   NaN   Stable →
6   Western Europe                   NaN   Stable →


In [5]:
# ─────────────────────────────────────────────
# DELIVERY RISK BY REGION
# ─────────────────────────────────────────────

region_df = df.groupby(['market', 'order_region']).agg(
    total_orders=('order_id', 'count'),
    late_orders=('late_delivery_risk', 'sum'),
    total_revenue=('sales', 'sum'),
    total_profit=('order_profit_per_order', 'sum'),
    revenue_at_risk=('sales', lambda x: x[df.loc[x.index, 'late_delivery_risk'] == 1].sum()),
    avg_delay_days=('shipping_delay_days', 'mean'),
    weighted_revenue_at_risk=('weighted_revenue_at_risk', 'sum')
).reset_index()

region_df['late_rate_pct'] = (region_df['late_orders'] / region_df['total_orders'] * 100).round(2)

region_df['late_rate_score']       = normalize(region_df['late_rate_pct'])
region_df['revenue_at_risk_score'] = normalize(region_df['revenue_at_risk'])
region_df['delay_score']           = normalize(region_df['avg_delay_days'])

region_df['profit_impact_score'] = normalize(
    pd.Series(
        np.where(region_df['total_profit'] < 0, region_df['total_profit'].abs(), 0),
        index=region_df.index
    )
)

region_df['volume_weight'] = normalize(region_df['total_orders'])

region_df['raw_risk_score'] = (
    (region_df['late_rate_score']       * 0.35) +
    (region_df['revenue_at_risk_score'] * 0.25) +
    (region_df['delay_score']           * 0.20) +
    (region_df['profit_impact_score']   * 0.20)
).round(2)

region_df['risk_score'] = (
    region_df['raw_risk_score'] * (0.7 + 0.3 * region_df['volume_weight'] / 100)
).round(2)

region_df['risk_flag'] = region_df['risk_score'].apply(risk_label)

# Merge trend
region_df = region_df.merge(trend_pivot, on='order_region', how='left')
region_df['risk_trend'] = region_df['risk_trend'].fillna('Stable →')
region_df['late_rate_change_pct'] = region_df['late_rate_change_pct'].fillna(0)

region_df = region_df.sort_values('risk_score', ascending=False)

print("Top 10 regions by risk score:")
print(region_df[['order_region', 'market', 'late_rate_pct',
                  'revenue_at_risk', 'risk_score', 'risk_flag', 'risk_trend']].head(10))

Top 10 regions by risk score:
       order_region        market  late_rate_pct  revenue_at_risk  risk_score  \
8    Western Europe        Europe          55.85     3.292014e+06       77.12   
10  Central America         LATAM          54.75     3.110260e+06       69.76   
11    South America         LATAM          54.31     1.606546e+06       47.90   
15       South Asia  Pacific Asia          56.27     8.714717e+05       47.46   
0    Central Africa        Africa          57.96     1.876883e+05       46.67   
16   Southeast Asia  Pacific Asia          55.53     1.073898e+06       45.19   
19      East of USA          USCA          55.66     7.540793e+05       43.67   
21       US Center           USCA          55.24     6.385042e+05       41.29   
14          Oceania  Pacific Asia          54.02     1.079244e+06       40.84   
6   Northern Europe        Europe          54.04     1.153863e+06       40.57   

      risk_flag risk_trend  
8     HIGH RISK   Stable →  
10  MEDIUM RISK   St

In [6]:
# ─────────────────────────────────────────────
# SHIPPING MODE RISK
# ─────────────────────────────────────────────

shipping_df = df.groupby('shipping_mode').agg(
    total_orders=('order_id', 'count'),
    late_orders=('late_delivery_risk', 'sum'),
    total_revenue=('sales', 'sum'),
    total_profit=('order_profit_per_order', 'sum'),
    revenue_at_risk=('sales', lambda x: x[df.loc[x.index, 'late_delivery_risk'] == 1].sum()),
    avg_delay_days=('shipping_delay_days', 'mean'),
    weighted_revenue_at_risk=('weighted_revenue_at_risk', 'sum')
).reset_index()

shipping_df['late_rate_pct'] = (shipping_df['late_orders'] / shipping_df['total_orders'] * 100).round(2)

shipping_df['late_rate_score']       = normalize(shipping_df['late_rate_pct'])
shipping_df['revenue_at_risk_score'] = normalize(shipping_df['revenue_at_risk'])
shipping_df['delay_score']           = normalize(shipping_df['avg_delay_days'])
shipping_df['profit_impact_score']   = normalize(
    pd.Series(
        np.where(shipping_df['total_profit'] < 0, shipping_df['total_profit'].abs(), 0),
        index=shipping_df.index
    )
)
shipping_df['volume_weight'] = normalize(shipping_df['total_orders'])

shipping_df['raw_risk_score'] = (
    (shipping_df['late_rate_score']       * 0.35) +
    (shipping_df['revenue_at_risk_score'] * 0.25) +
    (shipping_df['delay_score']           * 0.20) +
    (shipping_df['profit_impact_score']   * 0.20)
).round(2)

shipping_df['risk_score'] = (
    shipping_df['raw_risk_score'] * (0.7 + 0.3 * shipping_df['volume_weight'] / 100)
).round(2)

shipping_df['risk_flag'] = shipping_df['risk_score'].apply(risk_label)
shipping_df = shipping_df.sort_values('risk_score', ascending=False)

print("Shipping mode risk:")
print(shipping_df[['shipping_mode', 'late_rate_pct', 'revenue_at_risk',
                    'avg_delay_days', 'risk_score', 'risk_flag']])

Shipping mode risk:
    shipping_mode  late_rate_pct  revenue_at_risk  avg_delay_days  risk_score  \
2    Second Class          76.63     5.477446e+06        1.990828       53.63   
0     First Class          95.32     5.408069e+06        1.000000       53.02   
3  Standard Class          38.07     8.364603e+06       -0.004093       35.00   
1        Same Day          45.74     8.762769e+05        0.478279       13.67   

     risk_flag  
2  MEDIUM RISK  
0  MEDIUM RISK  
3     LOW RISK  
1     LOW RISK  


In [7]:
# ─────────────────────────────────────────────
# PRODUCT PROFITABILITY RISK
# ─────────────────────────────────────────────

product_df = df.groupby(['category_name', 'product_name']).agg(
    total_revenue=('sales', 'sum'),
    total_profit=('order_profit_per_order', 'sum'),
    total_quantity=('order_item_quantity', 'sum'),
    loss_orders=('is_loss_order', 'sum'),
    total_orders=('order_id', 'count')
).reset_index()

product_df['profit_margin_pct'] = (
    product_df['total_profit'] / product_df['total_revenue'] * 100
).round(2)

product_df['loss_rate_pct'] = (
    product_df['loss_orders'] / product_df['total_orders'] * 100
).round(2)

# 75th percentile threshold for volume trap
qty_threshold = product_df['total_quantity'].quantile(0.75)
print("Volume trap quantity threshold (75th percentile):", round(qty_threshold, 0))

def product_risk_tag(row):
    if row['total_profit'] < 0:
        return 'Loss Making'
    elif row['profit_margin_pct'] < 5 and row['total_quantity'] > qty_threshold:
        return 'Volume Trap'
    elif row['profit_margin_pct'] < 10:
        return 'Low Margin'
    else:
        return 'Healthy'

product_df['risk_tag'] = product_df.apply(product_risk_tag, axis=1)

print("\nProduct risk tag distribution:")
print(product_df['risk_tag'].value_counts())

print("\nTop 5 loss making products:")
print(product_df[product_df['risk_tag'] == 'Loss Making']
      .sort_values('total_profit')
      [['product_name', 'total_profit', 'profit_margin_pct']].head(5))

Volume trap quantity threshold (75th percentile): 888.0

Product risk tag distribution:
risk_tag
Healthy        75
Low Margin     40
Loss Making     3
Name: count, dtype: int64

Top 5 loss making products:
                               product_name  total_profit  profit_margin_pct
102                     SOLE E35 Elliptical   -965.119968              -3.22
77   Bushnell Pro X7 Jolt Slope Rangefinder   -255.950003              -3.88
13                      SOLE E25 Elliptical   -169.559997              -1.70


In [9]:
# Save all 3 risk files
region_df.to_csv('../data/cleaned/delivery_risk_by_region.csv', index=False)
shipping_df.to_csv('../data/cleaned/shipping_mode_risk.csv', index=False)
product_df.to_csv('../data/cleaned/product_profitability_risk.csv', index=False)

print("delivery_risk_by_region.csv saved — rows:", len(region_df))
print("shipping_mode_risk.csv saved — rows:", len(shipping_df))
print("product_profitability_risk.csv saved — rows:", len(product_df))

delivery_risk_by_region.csv saved — rows: 23
shipping_mode_risk.csv saved — rows: 4
product_profitability_risk.csv saved — rows: 118
